# Mixture of Experts (MoE)

> So far, every layer in our GPT has a single FFN shared by all tokens. Parameters and compute are locked together: double the parameters, and compute per token doubles too.
>
> This section introduces MoE. It splits one large FFN into multiple experts and activates only a few per token. Total parameters can be large while compute barely changes. We will build an MoE layer from scratch, focusing on how the router selects experts and how load is balanced.

The core idea of MoE: replace one FFN with several smaller expert FFNs, preceded by a router that decides which experts each token visits. For example, 8 experts with top-2 routing means total parameters grow 8x, but compute per token grows only 2x. More parameters means more knowledge capacity; compute stays roughly the same, so inference speed is unaffected.

After training for a while, the routing usually develops some structured division of labor: the Mixtral paper observed that adjacent tokens are often routed to the same expert, along with syntax-related routing patterns; DeepSeekMoE uses "fine-grained experts + shared experts" to push for more specialized division of labor. Note, however, that this does not mean each expert can simply be named a "grammar expert" or "numbers expert." Interpretability studies suggest that experts are more likely to learn fine-grained linguistic operations, local semantic patterns, or core capabilities that are commonly useful across domains. References: [Mixtral](https://embeddinglabs.com/papers/mixtral.pdf), [DeepSeekMoE](https://arxiv.org/abs/2401.06066), [MoE Interpretability](https://arxiv.org/abs/2604.02178), [Core Experts](https://huggingface.co/papers/2601.03425).

MoE has another headache: the router may get lazy, sending most tokens to a few experts while the rest sit idle. How to keep the load balanced is the central problem in MoE training.

## 1. The FFN Layer in a Standard Transformer

The fundamental difference between Dense and MoE models lies in the relationship between parameters and compute:

```
Dense model:
  All tokens -> same FFN -> output
  Parameters = compute per inference (all parameters participate)
  Want a stronger model -> enlarge FFN -> compute grows in lockstep

MoE model:
  Each token -> router picks top-k experts -> weighted output
  Total parameters = N x single-expert params (can be large)
  Compute per token = k x single-expert params (similar to Dense)
  Parameters and compute are decoupled
```

In a Dense model, giving the model more knowledge capacity means doubling the FFN dimension. But once the FFN grows, every token must pass through the larger FFN, so inference cost grows proportionally. MoE takes a different approach: distribute knowledge across multiple experts, activate only a few per token. Total parameters can be large, but compute per inference stays roughly constant.

Let us first review the standard FFN structure, then modify it into MoE.

Every Transformer Block contains an FFN (feed-forward network) with a simple structure: two linear transformations with an activation function in between:

```
Input x (d_model=512)
  |
Linear(512 -> 2048)  <- up-project, giving the model more 'thinking space'
  |
ReLU / GELU          <- nonlinearity, enabling complex pattern learning
  |
Linear(2048 -> 512)  <- down-project, back to the original dimension
  |
Output
```

Parameter count: 512 x 2048 + 2048 x 512 = ~2M (two matrices, ~1M each).

This FFN is one of the core sources of Transformer capability. Attention handles 'which positions to gather information from'; the FFN handles 'what to do with the gathered information.' Attention tells you 'which words this word is related to'; the FFN tells you 'given those relationships, what should this word become.'

But the current design has an implicit assumption: all tokens share the same FFN. Whether the input is a common function word or a technical term like 'quantum mechanics,' it passes through the same two matrices with the same transformation. This works fine in small models, but as the model grows, using one FFN to handle all types of knowledge becomes increasingly difficult, like using one set of rules to handle both grammar problems and math problems, the rules themselves become bloated.

### 1.1 Why MoE Usually Replaces the FFN Instead of Attention

You might ask: since a Transformer Block contains two major components, Attention and FFN, why does MoE typically target the FFN rather than splitting Attention into experts?

There are three reasons.

First, the FFN is computed per-token. Each token passing through the FFN is already like an independent sample: feed in a vector, get a vector out. The router can easily decide per token 'which experts to consult.'

Second, the FFN usually has a lot of parameters. The two or three large matrices of a Dense FFN account for a substantial fraction of the block's parameters. Splitting them into multiple experts can significantly increase model capacity, while each token only visits top-k experts, so compute does not scale linearly with the total number of experts.

Third, Attention is responsible for exchanging information between tokens. If Attention were also dynamically routed, the system would become more complex: tokens would not only need to pick experts but also look at each other, and communication and caching would become more cumbersome. Replacing the FFN first is a step with high payoff and a relatively clean change.

So you can think of MoE as: **keep the Attention information-exchange channel, and expand the FFN processing workshop behind it into multiple expert workshops.**

## 2. The Core Idea of MoE

If one FFN processing all tokens is too heavy a burden, we can take a different approach: split one large FFN into N smaller expert FFNs, and let each token consult only a few of them.

```
               +-------------+
               |    Router   |  <- decides which experts each token visits
               |   (Gate)    |
               +--+--+--+---+
                  |  |  |
          +-------+  |  +-------+
          v          v          v
     +--------++--------++--------+
     |Expert 1||Expert 2||Expert 3|  ... (8 total)
     | (FFN)  || (FFN)  || (FFN)  |
     +--------++--------++--------+
          |          |          |
          +----------+----------+
               weighted sum
```

The router takes the token's hidden state (a d_model-dimensional vector) as input and outputs N scores (one per expert). Experts with the highest scores are selected.

Each token activates only top-k experts (typically k=2), not all 8:

```
token 'the'     -> router -> experts 1, 5  (function word, handled by general experts)
token 'quantum' -> router -> experts 3, 7  (physics-related knowledge)
token 'hello'   -> router -> experts 2, 6  (English-related knowledge)
```

**Effect**:
- Total parameters = N x one expert's parameters (8x growth)
- Compute per inference = k x one expert's parameters (2x growth)
- **More parameters, less compute** <- parameters and compute are no longer locked together

The router itself is a trainable parameter matrix, `nn.Linear(d_model, num_experts)`. It is trained alongside the expert FFNs, learning through gradients how to select the most appropriate experts for each token.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MoELayer(nn.Module):
    """
    MoE FFN layer

    Args:
        d_model: hidden dimension
        num_experts: number of experts
        top_k: how many experts to activate per token
    """
    def __init__(self, d_model, num_experts=8, top_k=2, expert_dim=None):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        expert_dim = expert_dim or 4 * d_model

        # Router: input d_model, output num_experts scores
        self.gate = nn.Linear(d_model, num_experts, bias=False)

        # N experts, each is a small FFN
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, expert_dim),
                nn.ReLU(),
                nn.Linear(expert_dim, d_model)
            )
            for _ in range(num_experts)
        ])

    def forward(self, x):
        """
        x: [batch, seq_len, d_model]

        Steps:
        1. Router scores each token
        2. Select top-k experts
        3. Compute only the k selected experts' outputs
        4. Weighted sum
        """
        batch_size, seq_len, d_model = x.shape

        # Step 1: router scores
        gate_logits = self.gate(x)  # [batch, seq_len, num_experts]

        # Step 2: select top-k
        top_k_logits, top_k_indices = torch.topk(gate_logits, self.top_k, dim=-1)
        top_k_weights = F.softmax(top_k_logits, dim=-1)  # normalized weights

        # Step 3 & 4: for each token, compute selected experts' outputs and weighted sum
        output = torch.zeros_like(x)

        for b in range(batch_size):
            for s in range(seq_len):
                token = x[b, s]  # [d_model]
                for k in range(self.top_k):
                    expert_idx = top_k_indices[b, s, k].item()
                    weight = top_k_weights[b, s, k]
                    expert_out = self.experts[expert_idx](token.unsqueeze(0)).squeeze(0)
                    output[b, s] += weight * expert_out

        return output

print("MoE layer defined!")
print(f"8 experts, each token activates only 2")

In [ ]:
# Demonstrate MoE routing process
import torch
import torch.nn.functional as F

torch.manual_seed(42)

moe = MoELayer(d_model=16, num_experts=8, top_k=2)

# Simulate 3 tokens
x = torch.randn(1, 3, 16)

# Look at the router scores for each token
with torch.no_grad():
    gate_scores = moe.gate(x).squeeze(0)  # [3, 8]
    top_k_vals, top_k_idx = torch.topk(gate_scores, 2, dim=-1)

print("=== Experts selected by the router for 3 tokens ===")
print()
for i in range(3):
    experts = top_k_idx[i].tolist()
    weights = F.softmax(top_k_vals[i], dim=-1).tolist()
    print(f"Token {i}: selected experts {experts}, weights {[f'{w:.2f}' for w in weights]}")

print()
print("Each token activates only 2/8 = 25% of experts")
print("Total parameters are the sum of all 8 experts, but compute is only for 2 experts")

### 2.1 Comparison: Standard Transformer Block vs MoE Block

The best way to understand MoE's structure is not just through formulas, but by printing the model architecture directly.

The main pipeline of a standard decoder block is:

```text
x -> Attention -> FFN -> output
```

The main pipeline of a MoE decoder block is:

```text
x -> Attention -> Router -> select top-k from multiple FFN experts -> output
```

In other words, MoE does not replace Attention. Instead, it replaces the FFN in the Transformer Block with a 'router + multiple expert FFNs.'

In [ ]:
# Print real nn.Module structures: Dense FFN Block vs MoE Block
import torch.nn as nn

class TinyDenseBlock(nn.Module):
    """Skeleton of a standard Transformer decoder block: Attention + single FFN"""
    def __init__(self, d_model=16, num_heads=2, ffn_dim=64):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.ReLU(),
            nn.Linear(ffn_dim, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.self_attn(x, x, x, need_weights=False)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x

class TinyMoEBlock(nn.Module):
    """Skeleton of a MoE decoder block: Attention + Router + multiple FFN experts"""
    def __init__(self, d_model=16, num_heads=2, num_experts=4, top_k=2, expert_dim=64):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.moe = MoELayer(d_model, num_experts=num_experts, top_k=top_k, expert_dim=expert_dim)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.self_attn(x, x, x, need_weights=False)
        x = self.norm1(x + attn_out)
        moe_out = self.moe(x)
        x = self.norm2(x + moe_out)
        return x

dense_block = TinyDenseBlock()
moe_block = TinyMoEBlock(num_experts=4, top_k=2)

print("=== Standard Transformer Block ===")
print(dense_block)
print()
print("=== MoE Transformer Block ===")
print(moe_block)

Two things to notice in the printout above:

1. The standard block has only one `ffn`.
2. The MoE block has `moe` after `self_attn`, which contains `gate` and multiple `experts`.

This is the core structural evidence for MoE: **Attention is still there; the FFN has been replaced by multiple experts.**

In [ ]:
# Trace a forward pass: see shapes and where routing happens
import torch

trace_x = torch.randn(1, 3, 16)

print("=== Dense Block Trace ===")
with torch.no_grad():
    dense_attn_out, _ = dense_block.self_attn(trace_x, trace_x, trace_x, need_weights=False)
    dense_after_attn = dense_block.norm1(trace_x + dense_attn_out)
    dense_ffn_out = dense_block.ffn(dense_after_attn)
    dense_out = dense_block.norm2(dense_after_attn + dense_ffn_out)

print(f"input:        {tuple(trace_x.shape)}")
print(f"attention:    {tuple(dense_attn_out.shape)}")
print(f"single FFN:   {tuple(dense_ffn_out.shape)}")
print(f"output:       {tuple(dense_out.shape)}")

print()
print("=== MoE Block Trace ===")
with torch.no_grad():
    moe_attn_out, _ = moe_block.self_attn(trace_x, trace_x, trace_x, need_weights=False)
    moe_after_attn = moe_block.norm1(trace_x + moe_attn_out)
    gate_logits = moe_block.moe.gate(moe_after_attn)
    top_vals, top_idx = torch.topk(gate_logits, moe_block.moe.top_k, dim=-1)
    moe_out_raw = moe_block.moe(moe_after_attn)
    moe_out = moe_block.norm2(moe_after_attn + moe_out_raw)

print(f"input:        {tuple(trace_x.shape)}")
print(f"attention:    {tuple(moe_attn_out.shape)}")
print(f"gate logits:  {tuple(gate_logits.shape)}  # score per expert per token")
print(f"top-k index:  {tuple(top_idx.shape)}      # selected expert index per token")
print(f"MoE FFN:      {tuple(moe_out_raw.shape)}")
print(f"output:       {tuple(moe_out.shape)}")
print()
print("Experts selected for each token:")
for token_i, experts in enumerate(top_idx[0].tolist()):
    print(f"token {token_i}: experts {experts}")

### 2.2 Printing Real MoE Models from HuggingFace

Now that we understand the skeleton of a small model, let us look at a real decoder layer from production code. We are not downloading any weights, just using config to initialize a tiny Qwen2-MoE / Mixtral for one purpose: to print the module ordering from real source code.

When reading the printout, focus on three things:

1. `self_attn` is still at the front.
2. Where the standard FFN would be, there is now `mlp` / `SparseMoeBlock`.
3. Inside MoE, there is a `gate` and multiple `experts`.

Newer versions of `transformers` pack multiple expert weights into large tensors, so they may not display as `expert_0, expert_1, ...`. That is why, in addition to printing the layer, we also print parameter shapes: dimension 0 of `experts.gate_up_proj` is the expert count.

In [ ]:
# Print real HuggingFace MoE decoder layer, and trace a router output
import inspect
import warnings

import torch

warnings.filterwarnings("ignore", message="IProgress not found.*")

from transformers import Qwen2MoeConfig, Qwen2MoeForCausalLM
from transformers import MixtralConfig, MixtralForCausalLM

qwen_cfg = Qwen2MoeConfig(
    vocab_size=128,
    hidden_size=32,
    intermediate_size=64,
    moe_intermediate_size=64,
    shared_expert_intermediate_size=64,
    num_hidden_layers=1,
    num_attention_heads=4,
    num_key_value_heads=4,
    num_experts=4,
    num_experts_per_tok=2,
)
qwen_moe = Qwen2MoeForCausalLM(qwen_cfg)
qwen_layer = qwen_moe.model.layers[0]

print("=== Qwen2-MoE decoder layer ===")
print(qwen_layer)

print()
print("=== Qwen2-MoE MoE parameter shapes ===")
for name, param in qwen_layer.mlp.named_parameters():
    if name.startswith("gate") or name.startswith("experts") or name.startswith("shared"):
        print(f"{name:32s} {tuple(param.shape)}")

mixtral_cfg = MixtralConfig(
    vocab_size=128,
    hidden_size=32,
    intermediate_size=64,
    num_hidden_layers=1,
    num_attention_heads=4,
    num_key_value_heads=4,
    num_local_experts=4,
    num_experts_per_tok=2,
)
mixtral_moe = MixtralForCausalLM(mixtral_cfg)
mixtral_layer = mixtral_moe.model.layers[0]

print()
print("=== Mixtral decoder layer ===")
print(mixtral_layer)

print()
print("=== Mixtral MoE parameter shapes ===")
for name, param in mixtral_layer.mlp.named_parameters():
    if name.startswith("gate") or name.startswith("experts"):
        print(f"{name:32s} {tuple(param.shape)}")

print()
print("=== Qwen2-MoE layer.forward source excerpt ===")
source = inspect.getsource(type(qwen_layer).forward).splitlines()
for line in source:
    if "self_attn" in line or "mlp" in line or "layernorm" in line:
        print(line)

print()
print("=== Qwen2-MoE mlp.forward source excerpt ===")
source = inspect.getsource(type(qwen_layer.mlp).forward).splitlines()
for line in source:
    if "gate" in line or "expert" in line or "router" in line or "top" in line:
        print(line)

input_ids = torch.randint(0, 128, (1, 6))
with torch.no_grad():
    qwen_out = qwen_moe(input_ids=input_ids, output_router_logits=True)

router_logits = qwen_out.router_logits[0]
top_vals, top_idx = torch.topk(router_logits, qwen_cfg.num_experts_per_tok, dim=-1)

print()
print("=== Qwen2-MoE router trace ===")
print(f"input_ids:      {tuple(input_ids.shape)}")
print(f"logits:         {tuple(qwen_out.logits.shape)}")
print(f"router logits:  {tuple(router_logits.shape)}")
print(f"top-k experts:  {tuple(top_idx.shape)}")
print()
print("Experts selected for the first 6 tokens:")
for token_i, experts in enumerate(top_idx[:6].tolist()):
    print(f"token {token_i}: experts {experts}")

## 3. MoE Parameters vs Compute

This is MoE's core advantage. Let us feel it with concrete numbers:

Assume a Dense model's FFN has 2M parameters (d_model=512, d_ff=2048):

```
Standard Dense model:
  FFN parameters: 2M
  Compute per token: 2M parameter operations
  Parameters and compute are locked -> double params, double compute

MoE model (8 experts, top-2):
  FFN total params: 8 x 2M = 16M  <- 8x more parameters
  Compute per token: 2 x 2M = 4M  <- only 2x more compute
  Parameters and compute are decoupled
```

The actual computation path for a token in MoE looks like this:

1. Pass through the router: `W_gate @ x`, a small matrix multiplication (d_model x num_experts)
2. Pass through top-k expert FFNs: each expert internally does two matrix multiplications (up-project + down-project)
3. Weighted sum: combine the k experts' outputs using routing weights

The router's compute is much smaller than the FFN (d_model x num_experts << d_model x d_ff), so it is negligible. The main compute comes from the k expert FFNs.

This is why Mixtral 8x7B, despite having 47B total parameters, has inference speed comparable to a 7B Dense model: each inference activates only about 13B parameters (2 expert FFNs + shared Attention layers).

## 4. The Training Challenge: Load Balancing

MoE has an inherent engineering problem: **the router may get lazy and send tokens to only a few experts.**

Why does this happen? The router's only training objective is to minimize the next-token prediction loss. If the router discovers that sending all tokens to experts 3 and 5 produces low loss, it has no incentive to use other experts.

```
Bad case (imbalanced load):
  Expert 1: ====================  (overused)
  Expert 2: ====
  Expert 3: =
  Expert 4-8: (nearly idle, wasted parameters)

Good case (balanced load):
  Expert 1: ======
  Expert 2: ======
  Expert 3: ======
  ...
  Expert 8: ======
```

The consequences of load imbalance are severe: overused experts become bottlenecks (slower compute), idle experts learn nothing (wasted capacity), and the model degrades into a Dense model with only 2-3 effective experts.

**Traditional solution: auxiliary loss**

In addition to the language model's main loss, add an extra loss term that encourages each expert to process roughly the same number of tokens.

```python
# Load balance loss (simplified)
load_balance_loss = 0
for expert_i in range(num_experts):
    actual_load = count_tokens_assigned_to(expert_i)
    ideal_load = total_tokens * top_k / num_experts
    load_balance_loss += (actual_load - ideal_load) ** 2

total_loss = lm_loss + alpha * load_balance_loss
```

The coefficient alpha must be tuned manually. If alpha is too large, the router is forced to pick suboptimal experts (hurting model quality); if alpha is too small, load balancing is ineffective.

**Improved: Auxiliary-Loss-Free Load Balancing (DeepSeek-V3)**

The auxiliary loss approach works, but has an inherent contradiction: the auxiliary loss and the language model's main loss compete with each other. The auxiliary loss encourages uniform routing, while the main loss encourages sending tokens to the best experts. Balancing the two requires careful tuning of alpha: too large disrupts model convergence, too small leaves the load unbalanced.

DeepSeek-V3 proposed a more direct approach: instead of adding a loss to the router, directly adjust the router's bias. Each expert maintains a bias term $b_i$ added to the routing logits:

```
Normal routing: gate_logits = W_gate @ x
Improved:       gate_logits = W_gate @ x + b   (b does not participate in gradient computation)
```

The update rule for $b_i$ is simple: count how many tokens each expert processed this round. Experts whose load exceeds the mean have their bias decreased by gamma (causing them to receive fewer tokens next time); experts below the mean have their bias increased by gamma (causing them to receive more). Gamma is typically a small value (e.g., 0.001), and the biases naturally converge to a balanced point during training.

```python
# Executed after each training step
expert_loads = count_tokens_per_expert(selected_experts)
mean_load = total_tokens * top_k / num_experts

for i in range(num_experts):
    if expert_loads[i] > mean_load:
        bias[i] -= gamma   # overloaded -> lower bias
    else:
        bias[i] += gamma   # underloaded -> raise bias
```

Compared to auxiliary loss, this approach does not interfere with the main loss: bias updates are hand-crafted rules that produce no gradients and do not compete with the language model loss. The computational overhead is also near zero: one addition/subtraction per step, much cheaper than the backpropagation of an auxiliary loss.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MoELayerWithBias(nn.Module):
    """
    MoE layer with dynamic bias (DeepSeek-V3 style)

    Difference from standard MoE:
    - Added expert_bias parameter, added to routing logits
    - expert_bias is a buffer (no gradients), updated by hand-crafted rules
    - No longer needs auxiliary loss

    Args:
        d_model: hidden dimension
        num_experts: number of experts
        top_k: how many experts to activate per token
        gamma: bias adjustment step size (default 0.001)
    """
    def __init__(self, d_model, num_experts=8, top_k=2, gamma=0.001, expert_dim=None):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.gamma = gamma
        expert_dim = expert_dim or 4 * d_model

        self.gate = nn.Linear(d_model, num_experts, bias=False)
        # Key: one bias per expert, using register_buffer instead of Parameter
        # Buffers are not updated by optimizer; adjusted manually by update_bias
        self.register_buffer('expert_bias', torch.zeros(num_experts))

        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, expert_dim),
                nn.ReLU(),
                nn.Linear(expert_dim, d_model)
            )
            for _ in range(num_experts)
        ])

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape

        # Routing logits = linear transform + bias (bias does not participate in gradients)
        gate_logits = self.gate(x) + self.expert_bias

        top_k_logits, top_k_indices = torch.topk(gate_logits, self.top_k, dim=-1)
        top_k_weights = F.softmax(top_k_logits, dim=-1)

        output = torch.zeros_like(x)
        for b in range(batch_size):
            for s in range(seq_len):
                token = x[b, s]
                for k in range(self.top_k):
                    expert_idx = top_k_indices[b, s, k].item()
                    weight = top_k_weights[b, s, k]
                    expert_out = self.experts[expert_idx](token.unsqueeze(0)).squeeze(0)
                    output[b, s] += weight * expert_out

        return output, top_k_indices

    def update_bias(self, top_k_indices):
        """
        Update biases based on this round's routing results.
        Called after each optimizer.step().

        top_k_indices: [batch, seq_len, top_k]
        """
        total_tokens = top_k_indices.numel()  # batch * seq_len * top_k
        ideal_per_expert = total_tokens / self.num_experts

        # Count how many times each expert was selected
        expert_counts = torch.zeros(self.num_experts, device=self.expert_bias.device)
        for idx in top_k_indices.flatten():
            expert_counts[idx] += 1

        # Lower bias for overloaded experts, raise for underloaded
        for i in range(self.num_experts):
            if expert_counts[i] > ideal_per_expert:
                self.expert_bias[i] -= self.gamma
            elif expert_counts[i] < ideal_per_expert:
                self.expert_bias[i] += self.gamma
            # Equal to ideal: no change

print("MoE layer with dynamic bias defined!")
print("Key difference: gate_logits = W_gate @ x + expert_bias")
print("expert_bias is a buffer, not updated by optimizer, adjusted manually by update_bias()")

In [ ]:
# Demonstration: how dynamic bias drives load from imbalance toward balance
import torch

torch.manual_seed(42)

moe_bias = MoELayerWithBias(d_model=16, num_experts=8, top_k=2, gamma=0.1)

# Construct input: second half of tokens are scaled versions of the first half,
# inducing the router to concentrate on certain experts
x = torch.randn(1, 32, 16)
x[:, 16:, :] = x[:, :16, :] * 1.5

print("=== Bias convergence process ===")
print(f"Initial bias: {[f'{b:.2f}' for b in moe_bias.expert_bias.tolist()]}")
print()

num_steps = 30
for step in range(num_steps):
    _, top_k_idx = moe_bias(x)
    moe_bias.update_bias(top_k_idx)

    # Print load distribution every few steps
    if step < 5 or step % 10 == 0 or step == num_steps - 1:
        expert_counts = torch.zeros(8)
        for idx in top_k_idx.flatten():
            expert_counts[idx] += 1
        counts = expert_counts.int().tolist()
        imbalance = max(counts) - min(counts)
        print(f"Step {step:2d}: load distribution {counts}  range={imbalance}")

print()
print(f"Final bias: {[f'{b:.2f}' for b in moe_bias.expert_bias.tolist()]}")
print()
print("Key observations:")
print("1. Initially, load differences are large: some experts are overused, some nearly idle")
print("2. After each update_bias, overloaded experts' bias decreases, underloaded experts' bias increases")
print("3. After several steps, expert loads tend toward balance")
print("4. The entire process uses no auxiliary loss, relying purely on hand-crafted bias adjustments")

### 4.1 MoE Training Best Practices

When training MoE, the most common mistake is only monitoring `loss`. Loss going down does not mean MoE is training well: the router may use only a few experts, and the model appears to be learning while most experts receive insufficient gradients.

In practice, it is recommended to print four metrics every few steps:

| Metric | What to check | Warning sign |
|:---|:---|:---|
| `task_loss` | Is the main task being learned? | Loss not decreasing or severe jittering |
| `expert_usage` | Selection ratio per expert | One expert consistently at 80%+ |
| `imbalance` | Gap between busiest and idlest expert | Gap persistently large |
| `router_entropy` | Is the router becoming too rigid too early? | Low entropy, meaning selection is too peaked |

One-line takeaway: **Get the routing distribution healthy first, then worry about expert specialization.** DeepSeek-V3's auxiliary-loss-free load balancing serves the same goal: instead of using auxiliary loss to interfere with the main task, use dynamic expert bias to pull load back to a reasonable range.

More concretely, MoE training logs should track at least three lines simultaneously:

1. Is `task_loss` decreasing steadily?
2. Has `expert_usage` collapsed to a few experts over a long period?
3. Has `router_entropy` dropped too early?

If you only watch the first line, it is like only looking at the total exam score without checking who solved each problem. The problem with MoE is precisely that the total score may look fine, but many experts never participated in learning.

Reference: DeepSeek-V3 Technical Report, NVIDIA NeMo DeepSeek-V3 documentation, and 2025 follow-up work on expert load balancing all continue to treat routing collapse / load imbalance as the core problem of MoE.

In [ ]:
import torch
import torch.nn.functional as F

def router_metrics(gate_logits, top_k_indices, num_experts):
    """Print the routing metrics most worth monitoring during MoE training"""
    probs = F.softmax(gate_logits, dim=-1)
    entropy = -(probs * torch.log(probs + 1e-9)).sum(dim=-1).mean()

    counts = torch.bincount(top_k_indices.flatten(), minlength=num_experts).float()
    usage = counts / counts.sum()
    imbalance = counts.max() - counts.min()

    print("expert_usage:", [f"{u:.1%}" for u in usage.tolist()])
    print(f"imbalance: {imbalance.item():.0f} tokens")
    print(f"router_entropy: {entropy.item():.3f}")

    if usage.max() > 0.80:
        print("WARNING: Router may have collapsed; one expert is consuming 80%+ of routing.")
    elif imbalance > counts.mean():
        print("Note: Load is still skewed; consider monitoring bias / balance strategy.")
    else:
        print("Observation: Expert usage is relatively healthy; check if task_loss is decreasing.")

torch.manual_seed(42)
monitor_moe = MoELayerWithBias(d_model=16, num_experts=8, top_k=2, gamma=0.1)
monitor_x = torch.randn(1, 32, 16)
monitor_x[:, 16:, :] = monitor_x[:, :16, :] * 1.5

print("=== Before training: initial routing only ===")
with torch.no_grad():
    gate_logits = monitor_moe.gate(monitor_x) + monitor_moe.expert_bias
    _, top_k_idx = torch.topk(gate_logits, monitor_moe.top_k, dim=-1)
router_metrics(gate_logits, top_k_idx, monitor_moe.num_experts)

for _ in range(30):
    _, top_k_idx = monitor_moe(monitor_x)
    monitor_moe.update_bias(top_k_idx)

print("\n=== After dynamic bias adjustment: check routing again ===")
with torch.no_grad():
    gate_logits = monitor_moe.gate(monitor_x) + monitor_moe.expert_bias
    _, top_k_idx = torch.topk(gate_logits, monitor_moe.top_k, dim=-1)
router_metrics(gate_logits, top_k_idx, monitor_moe.num_experts)

print("\nKey observation: MoE training logs should consistently track expert_usage and router_entropy.")
print("Only watching task_loss may not reveal expert collapse.")

## 5. The Inference Challenge: All Experts Must Be Loaded

Although only k experts are activated per token, the router selects dynamically based on token content. Within the same batch, different tokens may select different expert combinations. Before inference begins, there is no way to know which experts will be selected.

This means all expert parameters must reside in GPU memory, waiting to be called:

```
Dense 7B:  memory ~ 14GB (FP16)
MoE 8x7B:  memory ~ 94GB (FP16) <- nearly 8x!
```

Even though each token only runs through 2 experts, all 8 experts' weights occupy GPU memory. This is MoE's trade-off: **low compute, but high memory.**

This is why MoE models, despite fast inference (low compute), require more GPUs, not because compute is insufficient, but because memory cannot fit everything. A MoE 8x7B model needs at least 2 A100s (80GB) to run, while a Dense 7B fits on one.

**Common mitigation strategies**:
- Quantization (INT4/INT8) compresses expert weights, reducing 94GB to ~24GB
- Offload rarely-used experts to CPU memory, loading them on demand (trading latency for memory)
- Expert Parallelism: distribute different experts across different GPUs; tokens are routed to the corresponding GPU via all-to-all communication

## 6. Notable MoE Models

| Model | Total Params | Active Params | Experts | Top-K | Notes |
|------|------------|-------------|--------|-------|------|
| **Mixtral 8x7B** | 47B | 13B | 8 | 2 | 2023, early representative open-source MoE |
| **DeepSeek-V2** | 236B | 21B | 160 | 6 | 2024, MLA + Shared Expert |
| **DeepSeek-V3** | 671B | 37B | 256 | 8 | 2024, auxiliary-loss-free load balancing / dynamic expert bias |
| **Qwen2.5-MoE** | 57B | 14B | 64 | 8 | 2024, shared expert + fine-grained experts |
| **GPT-4 (not public)** | undisclosed | undisclosed | undisclosed | undisclosed | closed-source model with no public full architecture; community rumors cannot be written as facts |

**Key number**: active params / total params depends on top-k, shared experts, attention, and non-expert parameters; it cannot be summarized by a single fixed ratio. The advantage of MoE is that total parameter capacity can be very large, but each token only activates part of the experts; actual speed also depends on batch, communication, memory bandwidth, and the inference framework.

From public models, a trend can be observed: researchers increasingly emphasize finer-grained experts, shared experts, and load balancing. But the number of experts, top-k, and activation ratio must be weighed together with communication overhead, batch shape, and memory capacity.

## 7. Why MoE Works: Separating Parameters from Compute

Recall the FFN in a Dense model. With d_model=4096, the FFN up-projects to 16384, with about 2 x 4096 x 16384 = ~134M parameters. Every token must pass through all 134M parameters, whether it is a simple function word or a complex term like 'quantum entanglement.'

MoE splits this apart. Assume 8 experts with equal parameters and top-k=2:

- Total parameters = 8 x 134M = ~1B (8x more)
- Compute per inference = 2 x 134M = ~268M (only 2x more)

Parameters increased 8x, but compute only 2x. The extra 6 experts' parameters are not wasted: they store more diverse knowledge, but do not need to be fully activated during inference.

More concretely, if during training the model discovers that certain knowledge combinations (e.g., 'French grammar' and 'C++ pointer syntax') rarely co-occur at the same token, these two types of knowledge can be assigned to different experts. The router learns to dispatch tokens to the correct expert based on their features: French tokens go to expert A, code tokens to expert B. This way, each expert's FFN can learn more specialized and deeper patterns, rather than one giant FFN trying to handle all types of knowledge simultaneously.

This is the fundamental difference between MoE and Dense: Dense binds parameters and compute together; MoE decouples them.

## 8. Advanced MoE Topics

**DeepSeek-V2's Innovation: Shared Expert + Routed Expert**

DeepSeek-V2 made an important structural improvement to MoE: in addition to top-k routed experts, it added a 'shared expert' that all tokens must pass through.

```
DeepSeek-V2 MoE:
  token -> shared expert (all tokens must pass) + routed experts top-k (selected as needed)
  
  Shared expert handles: grammatical structure, common vocabulary, commonsense reasoning (general capabilities all tokens need)
  Routed experts handle: math formulas, code syntax, domain-specific terminology (specialized knowledge)
```

Why this design? In standard MoE, general knowledge (e.g., 'how to organize a sentence') is learned separately by each expert, wasting parameters. Extracting general knowledge into a shared expert lets the routed experts focus on their specialties, improving parameter efficiency.

Qwen2-MoE also adopts a similar design. In the printed model structure, you can see `shared_expert` and `shared_expert_gate` components: the latter is a scalar gate that controls the output strength of the shared expert.

**Expert Parallelism**

When the number of experts is large (e.g., DeepSeek-V2's 160 experts), all expert parameters cannot fit on a single GPU. This requires Expert Parallelism: different experts are distributed across different GPUs. Tokens are sent to the corresponding GPU for FFN computation based on routing results, then returned to the original GPU.

This process requires all-to-all communication: each GPU may need to send tokens to every other GPU while receiving tokens from others. All-to-all communication is the primary overhead source in MoE training and inference, and is the core challenge in large-scale MoE system design.

**Fine-Grained Expert Segmentation**

DeepSeekMoE (2024) made a key finding: with total parameters and compute held constant, splitting each expert into more, smaller experts actually works better.

For example, an MoE layer with 8 experts at top-2 and another design with 64 smaller experts at top-8 can have identical total parameters and compute. The difference lies in the combinatorial selection space: C(8,2) = 28, while C(64,8) = 4.4 x 10^9. The latter can form many more expert combinations than the former, and each combination can learn more specialized knowledge.

The DeepSeekMoE paper reports the gains of fine-grained experts under its experimental settings; the specific granularity and savings ratio cannot be generalized apart from model size, data, and training configuration. Reference: [DeepSeekMoE](https://arxiv.org/abs/2401.06066).

**Choosing the Number of Experts and Top-K**

N (number of experts) and K (number activated) are not set arbitrarily; several common considerations drive them:

1. **Compute budget**. K x single-expert params should match the target FLOPs. Mixtral chooses top-2 so that per-token compute is close to 2 dense experts (~13B active params). DeepSeek-V3 chooses top-8, but each expert is smaller (FFN hidden dim 1408), keeping total active params around 37B.

2. **Fine-grained segmentation**. DeepSeekMoE's experiments show that fine-grained experts yield better parameter/compute efficiency under its settings; whether this always holds depends on the model, data, routing, and system implementation. The public designs of DeepSeekMoE / DeepSeek-V2 / DeepSeek-V3 reflect a direction toward finer-grained experts and shared experts; do not write about unofficially published follow-up models as confirmed configurations.

3. **Communication overhead**. In multi-GPU training, the larger K is, the more GPUs each token must be sent to for computation, and all-to-all communication grows linearly. DeepSeek-V3 introduces node-limited routing: each token crosses at most 4 nodes, so even with top-K=8 it only communicates with 4 nodes.

4. **Combinatorial diversity**. C(N,K) measures how many distinct expert combinations the model can form. The larger N is and the smaller K/N, the larger the combinatorial space. Let us compute the combination counts for several models with code below.

In [ ]:
import math

# MoE configuration comparison across models
configs = [
    ("Mixtral 8x7B", 8, 2),
    ("DeepSeek-V2", 160, 6),
    ("Qwen2.5-MoE", 64, 8),
    ("DeepSeek-V3", 256, 8),
]

print("=== MoE expert combination selection space C(N, K) ===")
print()
print(f"{'Model':20s} {'N':>4s} {'K':>3s} {'C(N,K)':>14s} {'Active ratio':>13s}")
for name, n, k in configs:
    c = math.comb(n, k)
    ratio = k / n * 100
    print(f"{name:20s} {n:4d} {k:3d} {c:14.2e} {ratio:12.1f}%")

print()
print("Key observations:")
print("1. Mixtral's C(8,2)=28, V4-Pro's C(384,6) = 6.6x10^11")
print("2. The combination space grows by more than 20 orders of magnitude -- the model has many more expert combinations available")
print("3. The active ratio drops from 25% (Mixtral) to 1.6% (V4-Pro) -- less activation, more choice")
print()

# V3 vs V4 comparison
print("=== How to interpret public MoE configurations ===")
print()
rows = [
    ("", "DeepSeek-V3", "Mixtral 8x7B"),
    ("Routed experts", "256", "8"),
    ("Top-K", "8", "2"),
    ("Shared expert", "yes", "no"),
    ("Public highlight", "load balancing / fine-grained experts", "simple, clean top-2 MoE"),
    ("Interpretation", "large-scale system engineering", "representative teaching intro"),
]
for label, v3, v4 in rows:
    print(f"{label:20s} {v3:>40s} {v4:>24s}")
print()
print("V4 increases the number of experts but lowers top-K -- a smaller active ratio, a larger combination space.")
print("In addition, V4 introduces CSA+HCA hybrid attention (supporting 1M context),")
print("the Muon optimizer (replacing AdamW), and FP4 quantization-aware training.")

---

## Summary

1. ✅ Dense FFN: all tokens share one set of parameters; parameter count = compute
2. ✅ MoE usually replaces the FFN instead of Attention: Attention handles information exchange, FFN handles per-token processing
3. ✅ MoE splits one large FFN into N expert FFNs + one router
4. ✅ The router scores each token, selects top-k experts, and outputs a weighted sum
5. ✅ Total parameters = N x single-expert params (can be large); compute per step = k x single-expert params
6. ✅ Parameters and compute are 'decoupled': total params can grow 8x while compute grows only 2x
7. ✅ Training challenge: load balancing -> auxiliary loss (traditional) or dynamic bias (DeepSeek-V3, does not interfere with main loss)
8. ✅ Inference challenge: all experts must be in GPU memory -> quantization / multi-GPU / expert parallelism
9. ✅ Notable MoE: Mixtral 8x7B (~47B total, ~13B active), large-scale MoE in public reports such as DeepSeek-V2/V3
10. ✅ The active ratio depends on top-k, shared experts, and non-expert parameters; it cannot be summarized as a fixed 10-20% for all MoE
11. ✅ Fine-grained expert segmentation: at equal FLOPs, more smaller experts beat fewer larger ones
12. ✅ The number of experts and Top-K are jointly constrained by compute budget, communication overhead, and combinatorial diversity

**One-line summary**: MoE decouples the parameters and compute that are bound together in Dense models: parameters store knowledge (more is better), compute determines inference speed (less is better). Load balancing is the core engineering challenge of this architecture, and DeepSeek-V3's dynamic bias approach makes auxiliary loss unnecessary. The public evolution of MoE points toward finer-grained experts, shared experts, load balancing, and better system implementations; do not treat rumors about unpublished models as facts.

## Exercises

> You may ask AI to help explain ideas, but it is not recommended to let AI directly 'solve these problems for you.'

**Exercise 1: MoE Parameter and Activation Count**

Mixtral 8x7B has 8 expert FFNs, each with about 7B parameters. Outside the MoE layer there are also ~1B shared Attention parameters. Compute the following two values:
1. The total parameter count of the MoE layer (sum of all 8 experts)
2. The parameter count actually activated per forward pass (top-2 routing, activating only 2 expert FFNs + shared Attention)

Hint: total params = shared params + 8 x single-expert params. Active params = shared params + 2 x single-expert params.

In [ ]:
# Exercise 1: MoE parameter and activation count
shared_params = 1e9   # shared Attention params (1B)
expert_params = 7e9   # per-expert FFN params (7B)
num_experts = 8
top_k = 2

# TODO: compute total MoE layer parameters
total_params = None  # compute here

# TODO: compute parameters activated per forward pass
active_params = None  # compute here

assert total_params is not None, "Please compute total params first"
assert active_params is not None, "Please compute active params first"

expected_total = shared_params + num_experts * expert_params
expected_active = shared_params + top_k * expert_params

assert total_params == expected_total, f"Total params should be {expected_total/1e9:.0f}B"
assert active_params == expected_active, f"Active params should be {expected_active/1e9:.0f}B"

print(f"✅ Exercise 1 passed:")
print(f"   Total params:      {total_params/1e9:.0f}B")
print(f"   Active params:     {active_params/1e9:.0f}B")
print(f"   Active ratio:      {active_params/total_params:.1%}")
print("   Core MoE advantage: total params can be large, but only a small portion is activated per pass.")

**Exercise 2: Load Balancing Analysis**

Suppose there are 4 experts processing 8 tokens. The router picks a top-1 expert for each token. Ideal load balancing means each expert handles 2 tokens. The actual routing result is: expert 0 handles 5 tokens, expert 1 handles 2, expert 2 handles 1, expert 3 handles 0.

Compute the routing entropy $H = -\sum_{i} p_i \log_2(p_i)$, where $p_i$ is the fraction of tokens handled by expert $i$. Compare it with the entropy of an ideal uniform distribution $H_{max} = \log_2(4)$.

Hint: under a uniform distribution $H = \log_2(N)$; the lower the entropy, the more imbalanced the load.

In [ ]:
# Exercise 2: load balancing analysis
import math

# Actual routing result
expert_counts = [5, 2, 1, 0]  # tokens handled per expert
total_tokens = sum(expert_counts)

# TODO: compute each expert's token fraction p_i
proportions = None  # compute here, list [p0, p1, p2, p3]

# TODO: compute routing entropy H = -sum(p_i * log2(p_i))
# Note: when p_i = 0, p_i * log2(p_i) = 0
entropy = None  # compute here

# Ideal maximum entropy
max_entropy = math.log2(4)

assert proportions is not None, "Please compute proportions first"
assert entropy is not None, "Please compute entropy first"

expected_props = [5/8, 2/8, 1/8, 0/8]
for i, (p, ep) in enumerate(zip(proportions, expected_props)):
    assert abs(p - ep) < 0.01, f"Expert {i}'s proportion should be {ep:.3f}, got {p:.3f}"

expected_H = -(5/8*math.log2(5/8) + 2/8*math.log2(2/8) + 1/8*math.log2(1/8) + 0)
assert abs(entropy - expected_H) < 0.01, f"Entropy should be {expected_H:.4f}, got {entropy:.4f}"

print(f"✅ Exercise 2 passed:")
print(f"   Actual entropy:    {entropy:.3f} bits")
print(f"   Max entropy:       {max_entropy:.3f} bits (uniform distribution)")
print(f"   Balance ratio:     {entropy/max_entropy:.1%}")
print("   The lower the entropy, the lazier the router -- sending most tokens to a few experts.")

**Exercise 3: MoE Inference Memory Estimation**

At MoE inference time, all experts' parameters must be loaded into GPU memory (even if only top-k are used each time). Suppose a MoE model has 1B shared Attention parameters, 8 expert FFNs of 7B each, and runs FP16 inference. Compute the GPU memory occupied by the model weights at inference (in GB; FP16 uses 2 bytes per parameter).

Hint: all parameters must be loaded at inference; total params x 2 bytes.

In [ ]:
# Exercise 3: MoE inference memory estimation
shared_params = 1e9       # 1B shared params
expert_params = 7e9       # 7B per expert
num_experts = 8
bytes_per_param = 2       # FP16

# TODO: compute total params
total_params = None  # compute here

# TODO: compute inference memory (GB)
memory_gb = None  # compute here

assert total_params is not None, "Please compute total params first"
assert memory_gb is not None, "Please compute memory first"

expected_total = shared_params + num_experts * expert_params
expected_mem = expected_total * bytes_per_param / 1e9

assert total_params == expected_total, f"Total params should be {expected_total/1e9:.0f}B"
assert abs(memory_gb - expected_mem) < 0.1, f"Memory should be {expected_mem:.1f} GB"

print(f"✅ Exercise 3 passed:")
print(f"   Total params:      {total_params/1e9:.0f}B")
print(f"   Inference memory:  {memory_gb:.1f} GB (model weights only, excluding KV cache)")
print("   MoE inference memory bottleneck: even if only 2 experts are active, all 8 must be loaded.")